# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*



## Finding 1 — The Anatomy of Growing Content

The paper reports that content trending upward differs structurally from content trending downward. Growing pages averaged about 3,180 words and 184 days old, while declining pages averaged about 2,311 words and 230 days old. The paper describes this as a directionally robust observational comparison, rather than causal evidence.

### My methodology question

How exactly is the `trend_direction` label constructed, and does the validation design ensure that the variables being compared are measured before the outcome?

The paper defines trend direction using the change in impressions between the most recent 30 days and the previous 30 days: `up` is greater than 10% growth, `down` is greater than 10% decline, `stable` is within +/-10%, while `flat` represents insufficient data and `new` represents content created within 30 days.

I would therefore ask whether all features used to characterize the growing and declining groups come from a period that is temporally prior to the trend label. If some features are measured from the same performance window used to construct the label, the relationship may partly reflect how the label itself was constructed rather than an independent predictive signal.

This does not invalidate the finding. It means the finding should be interpreted as an observed portfolio comparison unless a time-separated design establishes that the features precede the outcome.

---

## Finding 2 — The Content Performance Curve

The paper reports that content performance peaks around 61–90 days, declines after approximately 270 days, and shows a recovery among some 365+ day pages that were refreshed. The paper reports a health score of about 33 at 61–90 days and about 14 at 271–365 days.

### My methodology question

Does the comparison design separate the effect of content age from the effect of other factors that change with age, particularly freshness and survivor bias?

The paper itself notes that age and freshness are independent variables: old content can be freshly updated, while new content can already be stale. It also explicitly cautions that the 365+ long-unchanged group is a small active-content survivor sample and should not be treated as headline evidence.

I would therefore ask whether the age buckets contain comparable content populations and whether refreshed and unrefreshed pages are sufficiently comparable before interpreting the age pattern as a lifecycle effect.

A stronger validation design would control for or stratify by relevant factors such as freshness and compare similar content cohorts over time. This would make the evidence more supportive of a directional lifecycle interpretation rather than suggesting that age itself causes performance to decline.

---

## Methodology takeaway

Both findings are useful because they are based on observed portfolio patterns, but they also illustrate why validation design matters.

The paper itself describes the study as observational and states that correlations do not prove causation. Its machine-learning analyses are also described as exploratory and secondary to the direct portfolio evidence.

For my own model, I will apply the same principle: report what was measured on the evaluated split, test the model under a stricter validation design, audit possible leakage, and avoid causal or universal claims that the available evidence does not support.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*


### Before: Week-5 validation

In Week 5, the model was evaluated using a conventional random train/test split. The Random Forest achieved approximately 69.6% accuracy and a weighted F1 score of approximately 65.8%.

However, the dataset contains multiple content records belonging to the same `client_id`. A random row-level split can therefore place records from the same client in both the training and test sets.

This can make the test set less independent than intended and may give an optimistic estimate of generalization to a completely new client.

### After: Client-grouped validation

For this audit, I use `GroupShuffleSplit` with `client_id` as the grouping variable. This keeps all records belonging to a client entirely within either the training set or the test set.

The purpose is not to make the model look better or worse. The purpose is to measure how performance changes when the model is evaluated on clients it did not see during training.

The comparison uses the same target (`trend_direction`), the same Random Forest model, the same evaluation metrics, and a fixed random seed.

### Interpretation

If performance decreases under the grouped split, the difference suggests that the original random split may have benefited from client-level similarity between training and test observations.

The grouped result is therefore a more conservative estimate of how well the model may generalize to unseen clients.

This is an observed validation difference, not proof that the original model was invalid.

In [ ]:
# ============================================================
# W06 - SECTION 2
# My Model Under an Honest Split: Before vs After
# ============================================================

import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score


# ============================================================
# 1. LOAD DATA
# ============================================================

DATA_URL = (
    "https://raw.githubusercontent.com/"
    "acecod3z/Flyrankinternship/main/"
    "data/raw/content_refresh_anonymized.csv"
)

df = pd.read_csv(DATA_URL)

print("Dataset loaded successfully")
print("Shape:", df.shape)


# ============================================================
# 2. TARGET AND FEATURES
# ============================================================

y = df["trend_direction"]

drop_columns = [
    "content_id",
    "client_id",
    "trend_direction",
    "trend_pct"
]

X = df.drop(columns=drop_columns)

groups = df["client_id"]

print("\nTarget distribution:")
print(y.value_counts())


# ============================================================
# 3. DEFINE PREPROCESSING
# ============================================================

numeric_cols = X.select_dtypes(
    include=["number"]
).columns.tolist()

categorical_cols = X.select_dtypes(
    include=["object"]
).columns.tolist()

preprocessor = ColumnTransformer(
    transformers=[
        (
            "numeric",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="median")
                )
            ]),
            numeric_cols
        ),
        (
            "categorical",
            Pipeline([
                (
                    "imputer",
                    SimpleImputer(strategy="most_frequent")
                ),
                (
                    "onehot",
                    OneHotEncoder(
                        handle_unknown="ignore",
                        sparse_output=False
                    )
                )
            ]),
            categorical_cols
        )
    ]
)


# ============================================================
# 4. BEFORE — RANDOM TRAIN/TEST SPLIT
# ============================================================

X_train_before, X_test_before, y_train_before, y_test_before = (
    train_test_split(
        X,
        y,
        test_size=0.20,
        random_state=42,
        stratify=y
    )
)

rf_before = Pipeline([
    (
        "preprocessing",
        preprocessor
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_before.fit(
    X_train_before,
    y_train_before
)

pred_before = rf_before.predict(
    X_test_before
)

before_accuracy = accuracy_score(
    y_test_before,
    pred_before
)

before_f1 = f1_score(
    y_test_before,
    pred_before,
    average="weighted"
)

print("\n================ BEFORE ================")
print("Random split")
print("Accuracy:", round(before_accuracy, 4))
print("Weighted F1:", round(before_f1, 4))


# ============================================================
# 5. AFTER — CLIENT-GROUPED SPLIT
# ============================================================

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    splitter.split(
        X,
        y,
        groups=groups
    )
)

X_train_after = X.iloc[train_idx].copy()
X_test_after = X.iloc[test_idx].copy()

y_train_after = y.iloc[train_idx].copy()
y_test_after = y.iloc[test_idx].copy()


# ============================================================
# 6. TRAIN RANDOM FOREST ON GROUPED SPLIT
# ============================================================

rf_after = Pipeline([
    (
        "preprocessing",
        ColumnTransformer(
            transformers=[
                (
                    "numeric",
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(strategy="median")
                        )
                    ]),
                    numeric_cols
                ),
                (
                    "categorical",
                    Pipeline([
                        (
                            "imputer",
                            SimpleImputer(
                                strategy="most_frequent"
                            )
                        ),
                        (
                            "onehot",
                            OneHotEncoder(
                                handle_unknown="ignore",
                                sparse_output=False
                            )
                        )
                    ]),
                    categorical_cols
                )
            ]
        )
    ),
    (
        "model",
        RandomForestClassifier(
            n_estimators=200,
            random_state=42,
            n_jobs=-1
        )
    )
])

rf_after.fit(
    X_train_after,
    y_train_after
)

pred_after = rf_after.predict(
    X_test_after
)

after_accuracy = accuracy_score(
    y_test_after,
    pred_after
)

after_f1 = f1_score(
    y_test_after,
    pred_after,
    average="weighted"
)


# ============================================================
# 7. CLIENT OVERLAP CHECK
# ============================================================

train_clients = set(
    df.iloc[train_idx]["client_id"]
)

test_clients = set(
    df.iloc[test_idx]["client_id"]
)

client_overlap = train_clients.intersection(
    test_clients
)


print("\n================ AFTER ================")
print("Client-grouped split")
print("Accuracy:", round(after_accuracy, 4))
print("Weighted F1:", round(after_f1, 4))

print("\nTraining clients:", len(train_clients))
print("Testing clients :", len(test_clients))
print("Client overlap  :", len(client_overlap))


# ============================================================
# 8. BEFORE vs AFTER TABLE
# ============================================================

comparison = pd.DataFrame([
    {
        "Validation": "Week-5 Random Split",
        "Accuracy": before_accuracy,
        "Weighted F1": before_f1
    },
    {
        "Validation": "Week-6 Client-Grouped Split",
        "Accuracy": after_accuracy,
        "Weighted F1": after_f1
    }
])

print("\n================ COMPARISON ================")

display(
    comparison.round(4)
)

Dataset loaded successfully
Shape: (30000, 44)

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

================ BEFORE ================
Random split
Accuracy: 0.7875
Weighted F1: 0.7682

================ AFTER ================
Client-grouped split
Accuracy: 0.724
Weighted F1: 0.6935

Training clients: 25
Testing clients : 7
Client overlap  : 0

================ COMPARISON ================


,Validation,Accuracy,Weighted F1
0,Week-5 Random Split,0.7875,0.7682
1,Week-6 Client-Grouped Split,0.7240,0.6935


### Before / After Interpretation

The Week-5 model was evaluated using a random train/test split. In this audit, I repeated the evaluation using a client-grouped split.

The grouped split places all observations from a given client in only one of the two sets. The resulting client overlap was 0, confirming that the test clients were not present in the training set.

The difference between the two measured results shows how the validation design affects the estimated model performance.

The grouped result is the more conservative estimate when the intended use case involves generalization to clients that were not represented during training.

This comparison does not by itself prove that the original random split was invalid or that the model is overfitting. It shows that validation design changes the measured estimate of performance.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*



I reviewed the features used by the Week-5 model for possible target leakage.

The target is `trend_direction`. I excluded both `trend_direction` and `trend_pct` from the feature set because the project data documentation states that `trend_direction` is derived from `trend_pct`. Using either as a feature would therefore leak information from the target construction.

I also excluded `content_id` and `client_id`. These identifiers are useful for grouping and validation but should not be used as predictive features.

The remaining features are measurements and attributes available in the dataset, such as search volume, impressions, clicks, sessions, content age, freshness, CTR, average position, and engagement-related metrics.

For preprocessing, missing-value imputation and categorical encoding are performed inside the model pipeline. Therefore, preprocessing parameters are learned from the training portion rather than calculated from the complete dataset before splitting.

I did not use `trend_direction` or `trend_pct` as model inputs.

### Leakage decision

**No direct target leakage was intentionally included in the final feature set.**

The main validation improvement in this audit was the client-grouped split. This prevents records from the same client appearing in both training and testing and gives a more conservative estimate of performance on unseen clients.

In [ ]:
# ============================================================
# SECTION 3 — LEAKAGE AUDIT
# ============================================================

print("========== LEAKAGE AUDIT ==========\n")

target = "trend_direction"

# Columns that must not be features
leakage_columns = [
    "trend_direction",
    "trend_pct",
    "content_id",
    "client_id"
]

print("Target:")
print(" ", target)

print("\nColumns excluded from features:")
for col in leakage_columns:
    print(" ", col)


# ------------------------------------------------------------
# Check whether target/leakage columns are present in X
# ------------------------------------------------------------

leakage_in_features = [
    col for col in leakage_columns
    if col in X.columns
]

print("\nExcluded columns still present in X:")

if len(leakage_in_features) == 0:
    print("  NONE")
else:
    for col in leakage_in_features:
        print(" ", col)


# ------------------------------------------------------------
# Show final feature columns
# ------------------------------------------------------------

print("\nNumber of model features before encoding:")
print(" ", len(X.columns))

print("\nFinal feature columns:")

for col in X.columns:
    print(" ", col)


# ------------------------------------------------------------
# Explicit checks
# ------------------------------------------------------------

assert "trend_direction" not in X.columns
assert "trend_pct" not in X.columns
assert "content_id" not in X.columns
assert "client_id" not in X.columns

print("\nLeakage assertions:")
print("  trend_direction excluded: PASS")
print("  trend_pct excluded       : PASS")
print("  content_id excluded      : PASS")
print("  client_id excluded       : PASS")

print("\nOverall leakage audit: PASS")

========== LEAKAGE AUDIT ==========

Target:
  trend_direction

Columns excluded from features:
  trend_direction
  trend_pct
  content_id
  client_id

Excluded columns still present in X:
  NONE

Number of model features before encoding:
  40

Final feature columns:
  search_volume
  competition
  competition_level
  cpc
  content_type
  main_intent
  word_count
  char_count
  provider_used
  model_used
  impressions_90d
  clicks_90d
  pageviews_90d
  sessions_90d
  users_90d
  engaged_sessions_90d
  ai_sessions_90d
  scroll_events_90d
  days_with_impressions
  days_with_sessions
  impressions_last_30d
  clicks_last_30d
  sessions_last_30d
  impressions_prev_30d
  clicks_prev_30d
  sessions_prev_30d
  content_age_days
  age_tier
  age_tier_order
  days_since_last_update
  freshness_tier
  word_count_tier
  char_count_tier
  ctr
  avg_position
  engagement_rate
  scroll_rate
  ai_traffic_pct
  impression_tier
  position_tier

Leakage assertions:
  trend_direction excluded: PASS
  trend

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*



### Original claim

The Random Forest model achieves about 78.8% accuracy and therefore predicts content trend direction reliably.

### Revised claim

On the evaluated dataset, the Random Forest achieved **78.75% accuracy and 76.82% weighted F1** under the random split.

When evaluated using a **client-grouped split**, accuracy was **72.40%** and weighted F1 was **69.35%**.

The observed decrease under client-grouped validation indicates that the validation design affects the measured performance. The grouped result provides a more conservative estimate of performance when generalizing to clients not represented in training.

Therefore, the model should be treated as **decision-support**, rather than evidence that trend direction can be predicted reliably for every client.

The results are measured on this dataset and split and should not be interpreted as proof of performance on future clients or other datasets.

In [ ]:
# ============================================================
# SECTION 4 — CLAIM REWRITE
# ============================================================

print("========== CLAIM REWRITE ==========\n")

print("Original claim:")
print(
    "The Random Forest model achieves about 78.8% accuracy "
    "and therefore predicts content trend direction reliably."
)

print("\nRevised evidence-based claim:")

print(
    f"On the evaluated dataset, the Random Forest achieved "
    f"{before_accuracy:.2%} accuracy and "
    f"{before_f1:.2%} weighted F1 under the random split."
)

print(
    f"Under client-grouped validation, it achieved "
    f"{after_accuracy:.2%} accuracy and "
    f"{after_f1:.2%} weighted F1."
)

print(
    "\nThe grouped result is a more conservative estimate "
    "for generalization to unseen clients."
)

print(
    "The model is treated as decision-support rather than "
    "proof of reliable prediction for every client."
)

========== CLAIM REWRITE ==========

Original claim:
The Random Forest model achieves about 78.8% accuracy and therefore predicts content trend direction reliably.

Revised evidence-based claim:
On the evaluated dataset, the Random Forest achieved 78.75% accuracy and 76.82% weighted F1 under the random split.
Under client-grouped validation, it achieved 72.40% accuracy and 69.35% weighted F1.

The grouped result is a more conservative estimate for generalization to unseen clients.
The model is treated as decision-support rather than proof of reliable prediction for every client.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.